# Деревья решений — практическое задание

В этом ноутбуке рассматривается метод **деревьев решений** на примере набора данных **UCI Heart Disease Data** с Kaggle.
Ноутбук рассчитан на выполнение в **Google Colab**.

## Теоретическое введение

### 1. Постановка задачи классификации

Пусть имеется обучающая выборка
$$
\{(x_i, y_i)\}_{i=1}^n, \qquad x_i \in \mathbb{R}^d,\quad y_i \in \{0,1\}.
$$

Здесь:
- $x_i$ — вектор признаков объекта;
- $y_i$ — метка класса.

В задаче классификации нужно построить правило, которое по признакам объекта определяет, к какому классу он относится.

### 2. Идея дерева решений

**Дерево решений** — это модель, которая последовательно разбивает пространство признаков на области при помощи простых логических условий вида
$$
x_j \le t,
$$
где:
- $x_j$ — один из признаков;
- $t$ — пороговое значение.

Каждая внутренняя вершина дерева соответствует некоторому условию, а каждый лист — окончательному решению о классе объекта.

Таким образом, дерево реализует кусочно-постоянную аппроксимацию зависимости между признаками и целевой переменной.

### 3. Критерий качества разбиения

При построении дерева нужно выбрать, по какому признаку и в каком месте делать очередное разбиение. Для этого используют критерии неоднородности узла.

#### Индекс Джини

Для узла, в котором доля объектов класса $k$ равна $p_k$, индекс Джини определяется формулой
$$
G = 1 - \sum_{k=1}^{K} p_k^2.
$$

В задаче бинарной классификации:
$$
G = 1 - p_0^2 - p_1^2.
$$

Если в узле находятся объекты только одного класса, то неоднородность равна нулю.

#### Энтропия

Другой популярный критерий:
$$
H = - \sum_{k=1}^{K} p_k \log_2 p_k.
$$

Чем меньше неоднородность после разбиения, тем лучше выбран порог.

### 4. Рекурсивное построение дерева

Построение дерева выполняется рекурсивно:
1. для текущего узла перебираются возможные разбиения;
2. выбирается разбиение, которое сильнее всего уменьшает неоднородность;
3. процедура повторяется для дочерних узлов.

Этот процесс продолжается до выполнения условия остановки:
- достигнута максимальная глубина;
- в узле слишком мало объектов;
- дальнейшее разбиение не улучшает критерий.

### 5. Переобучение и ограничение сложности

Если позволить дереву расти без ограничений, оно может почти идеально запомнить обучающие данные. Это приводит к **переобучению**.

Для борьбы с переобучением используют параметры:
- `max_depth` — максимальная глубина дерева;
- `min_samples_split` — минимальное число объектов для разбиения;
- `min_samples_leaf` — минимальное число объектов в листе;
- `ccp_alpha` — параметр отсечения ветвей.

### 6. Интерпретация модели

Одно из преимуществ дерева решений — наглядность:
- можно визуализировать структуру дерева;
- можно определить важность признаков;
- можно проследить путь принятия решения.

Поэтому деревья решений полезны не только как практический метод классификации, но и как интерпретируемый инструмент анализа данных.


## Набор данных

Для работы используется набор данных о пациентах с признаками сердечно-сосудистых заболеваний:
[https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data).

Важно: в этом наборе данных целевая переменная называется **`num`**, а признак максимальной частоты сердечных сокращений — **`thalch`**. В ряде учебных материалов по тому же сюжету встречаются обозначения `target` и `thalach`, но в данном датасете используются именно имена `num` и `thalch`.

Целевая переменная:
- `num` — диагностический исход. Значение `0` означает отсутствие заболевания, а значения `1–4` соответствуют наличию заболевания разной степени выраженности. В задаче бинарной классификации удобно заменить этот столбец на бинарную переменную: `0` — нет заболевания, `1` — заболевание есть.

Примеры признаков:
- `age` — возраст;
- `sex` — пол;
- `cp` — тип боли в груди;
- `trestbps` — артериальное давление в покое;
- `chol` — уровень холестерина;
- `thalch` — максимальная достигнутая частота сердечных сокращений;
- `exang` — наличие стенокардии, вызванной нагрузкой;
- `oldpeak` — снижение сегмента ST;
- `ca`, `thal` — клинические характеристики обследования.

Этот набор данных хорошо подходит для деревьев решений, потому что:
- задача является задачей классификации;
- признаки имеют медицинский смысл;
- дерево решений может находить нелинейные зависимости и пороговые правила;
- важность признаков удобно интерпретировать.

Самостоятельно выберите способ загрузить данные и сделать их доступными в блокноте с заданием.

Один из вариантов — скачивание через Kaggle API, используя свой токен. Ещё один вариант — скачать файл вручную, выложить его на Google-диск и потом подмонтировать Google-диск в среду Colab. Ниже приведены инструкции.



## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную настройку Kaggle и Colab (делается один раз).
1. Откройте https://www.kaggle.com, зайдите в свой профиль, затем в настройках профиля найдите раздел **API Tokens**.
2. Создайте токен и сразу же скопируйте его в текстовый документ на своём локальном компьютере.
3. Там же посмотрите, под каким именем пользователя (`username`) вы работаете в Kaggle. Скопируйте его в тот же документ.
4. В Colab откройте левую панель, вкладку **Secrets**.
5. Создайте два секрета:
- `KAGGLE_USERNAME` : вставьте username
- `KAGGLE_KEY`      : вставьте token

Не забудьте установить переключатель **Доступ из блокнотов**.

Далее следуйте инструкциям в коде ниже:


In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
!kaggle datasets list -s heart disease | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml"
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

# Настройка адреса скачивания: берём из URL всё, что идёт
# после /datasets/ : https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data
DATASET = "redwankarimsony/heart-disease-data"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
print("Содержимое папки:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)


## Альтернативный вариант — скачивание файла вручную

- создайте у себя на Google-диске папку `sci-prog-and-ml`;
- скачайте файл данных вручную и поместите его в эту папку на Google-диске;
- подмонтируйте Google-диск к среде Colab, выполнив код ниже;

После выполнения этих инструкций папка `sci-prog-and-ml` будет сделана рабочей, и из неё уже можно будет открывать файл при помощи `df = pd.read_csv(file_name)`.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')
print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)


## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Загрузите файл с данными в `pandas.DataFrame`.
2. Выведите первые строки таблицы.
3. Определите:
   - размер таблицы;
   - список признаков;
   - наличие пропущенных значений;
   - распределение целевой переменной.

### Часть 2. Подготовка данных
1. Выберите признаки для первого эксперимента:
   - `age`,
   - `sex`,
   - `cp`,
   - `trestbps`,
   - `chol`,
   - `fbs`,
   - `restecg`,
   - `thalach`,
   - `exang`,
   - `oldpeak`,
   - `slope`,
   - `ca`,
   - `thal`.
2. Оставьте целевую переменную `target`.
3. Если в данных имеются пропуски, обработайте их подходящим способом.
4. Разделите данные на признаки `X` и целевую переменную `y`.

### Часть 3. Построение модели
1. Разделите выборку на обучающую и тестовую части.
2. Создайте модель дерева решений для классификации.
3. Подберите разумные параметры, например `max_depth`.
4. Обучите модель на обучающей выборке.
5. Получите предсказания на тестовой выборке.

### Часть 4. Оценка качества и интерпретация
1. Вычислите `accuracy`.
2. Постройте `confusion matrix`.
3. Выведите `classification report`.
4. Найдите важности признаков (`feature_importances_`).
5. Визуализируйте дерево решений.
6. Сделайте вывод: какие признаки оказались наиболее важными и не слишком ли сложным получилось дерево.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import pandas as pd
import numpy as np
```

### Загрузка данных

После распаковки датасета в рабочей папке обычно появляется CSV-файл с медицинскими данными.
Чтобы код не зависел от точного имени файла, удобно автоматически найти первый CSV-файл:

```python
csv_files = glob.glob("*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
```

### Проверка пропусков
```python
df.isna().sum()
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

### Модель дерева решений
```python
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
```

### Получение предсказаний
```python
y_pred = model.predict(X_test)
```

### Оценка качества
```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

### Важности признаков
```python
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)
display(importance)
```

### Визуализация дерева
```python
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 8))
plot_tree(model, feature_names=X.columns, class_names=["0", "1"], filled=True)
plt.show()
```


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. Почему дерево решений удобно интерпретировать по сравнению со многими другими моделями?
2. Почему слишком глубокое дерево может переобучиться?
3. Чем критерий Джини отличается от энтропии на практическом уровне?
4. Почему важно проверять качество модели именно на тестовой выборке?
5. Какие признаки в данном наборе данных оказались наиболее важными и почему это может быть содержательно разумно?


## Дополнительные задания

### Задание 1
Измените параметр `max_depth` и сравните качество моделей при нескольких значениях глубины.

### Задание 2
Попробуйте изменить критерий разбиения:
- `criterion="gini"`;
- `criterion="entropy"`.

Сравните результаты.

### Задание 3
Подберите параметры `min_samples_split` и `min_samples_leaf`. Посмотрите, как это влияет на структуру дерева и качество модели.

### Задание 4
Сравните дерево решений с логистической регрессией на этом же наборе данных. Обсудите, какая модель лучше интерпретируется и какая показывает лучшее качество.
